In [1]:
# ============================================================
# PHASE 9: LLM REPORT GENERATION
# Goal: Generate structured investigation report per lot
# LLM: Groq (Llama 3) — fast, free
# Input: SHAP + Hypothesis + DiCE + Anomaly per lot
# ============================================================

import pandas as pd
import numpy as np
import json
import os
import warnings
warnings.filterwarnings('ignore')

OUTPUT_DIR = r'C:\Users\ASUS\Downloads\capstone\Project\outputs'

In [2]:
pip install groq

Note: you may need to restart the kernel to use updated packages.


In [3]:
# ============================================================
# RELOAD ALL PHASE OUTPUTS
# ============================================================

flagged       = pd.read_csv(f'{OUTPUT_DIR}/data/p3_flagged_lots.csv')
per_lot_shap  = pd.read_csv(f'{OUTPUT_DIR}/data/p4_per_lot_shap.csv')
top25         = pd.read_csv(f'{OUTPUT_DIR}/data/p4_top25_stable_sensors.csv')
hypotheses    = pd.read_csv(f'{OUTPUT_DIR}/data/p6_hypotheses.csv')
csr_df        = pd.read_csv(f'{OUTPUT_DIR}/data/p7_csr_results.csv')
sensor_changes= pd.read_csv(f'{OUTPUT_DIR}/data/p7_sensor_changes.csv')
lot_summary   = pd.read_csv(f'{OUTPUT_DIR}/data/p8_lot_summary.csv')
stability_df  = pd.read_csv(f'{OUTPUT_DIR}/data/p4_bootstrap_stability.csv')

print(f"✅ All data loaded")
print(f"   Flagged lots : {len(flagged)}")
print(f"   Fail lots    : {flagged['true_label'].eq(1).sum()}")

✅ All data loaded
   Flagged lots : 8
   Fail lots    : 5


In [ ]:
# ============================================================
# SETUP GROQ CLIENT
# ============================================================

from groq import Groq

GROQ_API_KEY = ""  # paste your key here

client = Groq(api_key=GROQ_API_KEY)

# Test connection
test = client.chat.completions.create(
    model    = "llama-3.1-8b-instant",
    messages = [{"role": "user", "content": "Say: Groq connected"}],
    max_tokens = 20
)
print(f"✅ Groq connected : {test.choices[0].message.content}")

✅ Groq connected : Groq is a company known for its Tensor Processing Units (TPUs) and AI hardware, often


In [5]:
# ============================================================
# STEP 9.1 — BUILD STRUCTURED CONTEXT PER LOT
# This is the prompt data — what we feed to LLM
# ============================================================

def build_lot_context(lot_idx):
    lot_row  = flagged[flagged['lot_index'] == lot_idx].iloc[0]
    hyp_row  = hypotheses[hypotheses['lot_index'] == lot_idx].iloc[0]
    lot_shap = per_lot_shap[
        per_lot_shap['lot_index'] == lot_idx
    ].head(10)
    csr_row  = csr_df[csr_df['lot_index'] == lot_idx]
    changes  = sensor_changes[
        sensor_changes['lot_index'] == lot_idx
    ].groupby('sensor')['change'].mean().sort_values()

    # Top 5 SHAP sensors
    top5_shap = lot_shap.head(5)[['sensor','shap_value']].to_dict('records')

    # Top 5 DiCE suggestions
    top5_dice = [
        {'sensor': s, 'suggested_change': round(float(v), 4)}
        for s, v in changes.head(5).items()
    ] if len(changes) > 0 else []

    context = {
        'lot_index':         lot_idx,
        'true_label':        'FAIL' if lot_row['true_label'] == 1 else 'PASS',
        'risk_tier':         lot_row['risk_tier'],
        'hybrid_score':      round(lot_row['hybrid_score'], 4),
        'primary_candidate': hyp_row['primary_candidate'],
        'confidence':        hyp_row['confidence'],
        'deviation_sigma':   round(hyp_row['deviation_sigma'], 4),
        'top3_candidates':   hyp_row['top3_sensors'],
        'top5_shap':         top5_shap,
        'top5_dice':         top5_dice,
        'csr': round(csr_row['csr'].values[0], 4) \
               if len(csr_row) > 0 else 'N/A',
        'pcr': round(csr_row['pcr'].values[0], 4) \
               if len(csr_row) > 0 else 'N/A'
    }
    return context

# Test on first fail lot
first_fail = flagged[flagged['true_label'] == 1].iloc[0]['lot_index']
# Fix numpy types for JSON serialization
def convert(obj):
    if isinstance(obj, (np.integer, np.int64)):
        return int(obj)
    if isinstance(obj, (np.floating, np.float64)):
        return float(obj)
    return obj

ctx = build_lot_context(int(first_fail))
print(f"✅ Context built for Lot {first_fail}")
print(json.dumps(ctx, indent=2, default=convert))

✅ Context built for Lot 1
{
  "lot_index": 1,
  "true_label": "FAIL",
  "risk_tier": "High Risk",
  "hybrid_score": 0.6888,
  "primary_candidate": 59,
  "confidence": "HIGH",
  "deviation_sigma": 2.3071,
  "top3_candidates": "['59', '426', '420']",
  "top5_shap": [
    {
      "sensor": 44,
      "shap_value": 0.0207618910273564
    },
    {
      "sensor": 133,
      "shap_value": 0.0205241919629362
    },
    {
      "sensor": 59,
      "shap_value": 0.0194153325321136
    },
    {
      "sensor": 45,
      "shap_value": 0.0148635700949776
    },
    {
      "sensor": 123,
      "shap_value": 0.0147702310086974
    }
  ],
  "top5_dice": [
    {
      "sensor": 485,
      "suggested_change": 0.5996
    },
    {
      "sensor": 468,
      "suggested_change": 1.4844
    },
    {
      "sensor": 519,
      "suggested_change": 4.9543
    },
    {
      "sensor": 460,
      "suggested_change": 8.2355
    },
    {
      "sensor": 420,
      "suggested_change": 15.1816
    }
  ],
  "csr": 0.

In [6]:
# ============================================================
# STEP 9.2 — LLM REPORT GENERATOR
# Structured prompt → structured report
# ============================================================

SYSTEM_PROMPT = """You are an expert semiconductor process engineer 
assistant analyzing wafer yield failures in a SECOM manufacturing dataset.

You receive structured data about a flagged production lot and generate
a professional failure investigation report.

CRITICAL RULES:
1. Sensors are anonymous (numbered) — never invent process names
2. Always say "candidate sensor" not "root cause"
3. Always say "statistical hypothesis" not "confirmed cause"
4. Be concise — engineers are busy
5. Always end with actionable investigation steps
6. Format: use the exact sections provided

Your report must follow this structure:
## LOT [ID] FAILURE INVESTIGATION REPORT
### 1. Executive Summary (2-3 sentences)
### 2. Risk Assessment
### 3. Primary Investigation Candidate
### 4. Supporting Evidence
### 5. Suggested Process Adjustments
### 6. Investigation Action Items
### 7. Confidence Statement
"""

def generate_report(lot_idx):
    ctx = build_lot_context(lot_idx)

    user_prompt = f"""
Generate a failure investigation report for this lot:

LOT DATA:
- Lot Index       : {ctx['lot_index']}
- True Label      : {ctx['true_label']}
- Risk Tier       : {ctx['risk_tier']}
- Anomaly Score   : {ctx['hybrid_score']}
- Primary Sensor  : {ctx['primary_candidate']}
- Confidence      : {ctx['confidence']}
- Deviation       : {ctx['deviation_sigma']}σ from normal

TOP 5 SHAP SENSORS (by impact on failure prediction):
{json.dumps(ctx['top5_shap'], indent=2)}

TOP DiCE SUGGESTED ADJUSTMENTS (to achieve PASS):
{json.dumps(ctx['top5_dice'], indent=2)}

CORRECTION METRICS:
- CSR (full flip rate) : {ctx['csr']}
- PCR (gap closed)     : {ctx['pcr']}

Generate the structured report now.
"""

    response = client.chat.completions.create(
        model    = "llama-3.1-8b-instant",
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_prompt}
        ],
        max_tokens  = 800,
        temperature = 0.3  # low temp = consistent, factual output
    )

    return response.choices[0].message.content

# Test on first fail lot
print(f"Generating report for Lot {first_fail}...")
report = generate_report(int(first_fail))
print(report)

Generating report for Lot 1...
## LOT 1 FAILURE INVESTIGATION REPORT

### 1. Executive Summary
A high-risk production lot (Lot Index: 1) failed to meet quality standards, with a True Label of FAIL and a significant Anomaly Score of 0.6888. The investigation aims to identify the primary cause of failure.

### 2. Risk Assessment
The lot is classified as High Risk, indicating a high likelihood of production issues. The Anomaly Score further emphasizes the severity of the failure, with a deviation of 2.3071σ from normal.

### 3. Primary Investigation Candidate
Based on the SHAP values, the primary investigation candidate is Sensor 59 (SHAP value: 0.0194153325321136). This sensor is among the top 3 most impactful features in predicting the failure.

### 4. Supporting Evidence
The DiCE suggested adjustments highlight potential process changes that could lead to a PASS outcome. Notably, Sensors 485, 468, 519, 460, and 420 have suggested changes ranging from 0.5996 to 15.1816. However, the pri

In [7]:
# ============================================================
# STEP 9.3 — GENERATE REPORTS FOR ALL FLAGGED LOTS
# ============================================================

all_reports = {}
fail_lots   = flagged[flagged['true_label'] == 1]['lot_index'].tolist()

print(f"Generating reports for {len(fail_lots)} fail lots...\n")

for lot_idx in fail_lots:
    print(f"{'='*60}")
    print(f"LOT {lot_idx}")
    print(f"{'='*60}")
    report = generate_report(int(lot_idx))
    all_reports[lot_idx] = report
    print(report)
    print()

print(f"\n✅ {len(all_reports)} reports generated")

Generating reports for 5 fail lots...

LOT 1
## LOT 1 FAILURE INVESTIGATION REPORT

### 1. Executive Summary (2-3 sentences)
A high-risk production lot (Lot Index 1) failed with a high anomaly score of 0.6888 and a deviation of 2.3071σ from normal. The primary sensor contributing to the failure is candidate sensor 59, with a SHAP value of 0.0194153325321136. The investigation aims to identify the underlying cause of this failure.

### 2. Risk Assessment
The lot is classified as High Risk, indicating a significant likelihood of yield loss. The high anomaly score and deviation from normal further emphasize the need for a thorough investigation.

### 3. Primary Investigation Candidate
Candidate sensor 59 is the primary sensor contributing to the failure, with a SHAP value indicating its significant impact on the failure prediction. This sensor is a critical component in the manufacturing process, and its performance may be a key factor in the lot's failure.

### 4. Supporting Evidence
The

In [8]:
# ============================================================
# STEP 9.4 — REPORT QUALITY SCORING
# Ask LLM to score each report on 5 criteria
# ============================================================

def score_report(lot_idx, report):
    score_prompt = f"""
Rate this failure investigation report on these 5 criteria.
Return ONLY a JSON object, nothing else.

REPORT:
{report}

Rate each criterion from 1-5:
1. clarity         — Is the report clear and well structured?
2. actionability   — Does it give clear investigation steps?
3. honesty         — Does it avoid overclaiming causation?
4. completeness    — Does it cover all key findings?
5. engineer_value  — Would a process engineer find this useful?

Return exactly this format:
{{"clarity": X, "actionability": X, "honesty": X, "completeness": X, "engineer_value": X, "overall": X, "comment": "one sentence"}}
"""

    response = client.chat.completions.create(
        model    = "llama-3.1-8b-instant",
        messages = [{"role": "user", "content": score_prompt}],
        max_tokens  = 200,
        temperature = 0.1
    )

    raw = response.choices[0].message.content
    try:
        # Strip markdown if present
        clean = raw.replace('```json','').replace('```','').strip()
        scores = json.loads(clean)
        return scores
    except:
        print(f"  ⚠️  Score parse failed for lot {lot_idx}")
        return {
            'clarity': 3, 'actionability': 3, 'honesty': 3,
            'completeness': 3, 'engineer_value': 3,
            'overall': 3, 'comment': 'parse error'
        }

# Score all reports
score_results = []
for lot_idx, report in all_reports.items():
    scores = score_report(lot_idx, report)
    scores['lot_index'] = lot_idx
    score_results.append(scores)
    print(f"Lot {lot_idx} — Overall: {scores.get('overall', 'N/A')}/5 | {scores.get('comment','')}")

scores_df    = pd.DataFrame(score_results)
overall_qual = scores_df['overall'].mean()

print(f"\n{'='*50}")
print(f"  REPORT QUALITY SUMMARY")
print(f"{'='*50}")
print(scores_df[[
    'lot_index','clarity','actionability',
    'honesty','completeness','engineer_value','overall'
]].to_string(index=False))
print(f"\n  Mean Quality Score : {overall_qual:.2f} / 5.0")
print(f"  Target             : > 3.5 / 5.0")
print(f"  Status             : {'✅ HIT' if overall_qual >= 3.5 else '⚠️ Below target'}")

Lot 1 — Overall: 4.4/5 | The report is well-structured and provides actionable steps, but could benefit from more detailed analysis and data to support its findings.
Lot 97 — Overall: 4.6/5 | The report is well-structured and provides actionable steps, but could benefit from more detailed analysis of sensor 59's behavior in previous production lots.
Lot 105 — Overall: 4.6/5 | The report is well-structured and provides actionable steps, but could benefit from more detailed analysis and supporting evidence.
Lot 118 — Overall: 4.4/5 | The report is well-structured and provides actionable steps, but could benefit from more detailed explanations of the statistical methods used.
Lot 123 — Overall: 4.6/5 | The report is well-structured and provides actionable steps, but could benefit from more detailed explanations of the analysis and results.

  REPORT QUALITY SUMMARY
 lot_index  clarity  actionability  honesty  completeness  engineer_value  overall
         1        5              4        

In [9]:
# ============================================================
# STEP 9.5 — SAVE ALL OUTPUTS
# ============================================================

import os

# Save reports as text files
reports_dir = f'{OUTPUT_DIR}/reports'
os.makedirs(reports_dir, exist_ok=True)

for lot_idx, report in all_reports.items():
    with open(f'{reports_dir}/lot_{lot_idx}_report.txt', 'w') as f:
        f.write(report)

# Save scores
scores_df.to_csv(
    f'{OUTPUT_DIR}/data/p9_report_scores.csv', index=False)

# Save all reports as single JSON
with open(f'{OUTPUT_DIR}/data/p9_all_reports.json', 'w') as f:
    json.dump(all_reports, f, indent=2)

with open(f'{OUTPUT_DIR}/models/report_quality_score.txt', 'w') as f:
    f.write(str(overall_qual))

print(f"✅ {len(all_reports)} report files saved → outputs/reports/")
print(f"✅ p9_report_scores.csv    saved")
print(f"✅ p9_all_reports.json     saved")
print(f"✅ report_quality_score    saved ({overall_qual:.2f})")
print(f"\n🚀 PHASE 9 DONE — Ready for Phase 10 (Streamlit)")
print(f"   Reports generated : {len(all_reports)}")
print(f"   Quality Score     : {overall_qual:.2f} / 5.0")
print(f"\n   All reports → outputs/reports/")
print(f"   These reports → Phase 10 Streamlit dashboard")

✅ 5 report files saved → outputs/reports/
✅ p9_report_scores.csv    saved
✅ p9_all_reports.json     saved
✅ report_quality_score    saved (4.52)

🚀 PHASE 9 DONE — Ready for Phase 10 (Streamlit)
   Reports generated : 5
   Quality Score     : 4.52 / 5.0

   All reports → outputs/reports/
   These reports → Phase 10 Streamlit dashboard


In [10]:
# ============================================================
# OBJECTIVE EVALUATION CHECKLIST
# Removes "self-grading" concern
# ============================================================

checklist_results = []

for lot_idx, report in all_reports.items():
    r = report.lower()

    checks = {
        'has_primary_sensor':    'sensor' in r and 'candidate' in r,
        'has_deviation_value':   'sigma' in r or 'deviation' in r or 'σ' in r,
        'has_correction':        'adjust' in r or 'reduce' in r or 'decreas' in r,
        'avoids_root_cause':     'root cause' not in r,
        'has_action_items':      'action' in r or 'step' in r or 'inspect' in r,
        'has_confidence_stmt':   'hypothesis' in r or 'statistical' in r or 'validation required' in r,
        'has_7_sections':        all(
            kw in r for kw in [
                'executive', 'risk', 'candidate',
                'evidence', 'adjustment', 'action', 'confidence'
            ]
        ),
    }

    score = sum(checks.values())
    checklist_results.append({
        'lot_index': lot_idx,
        **{k: '✓' if v else '✗' for k, v in checks.items()},
        'total_score': f"{score}/7"
    })

checklist_df = pd.DataFrame(checklist_results)
print("\nOBJECTIVE REPORT CHECKLIST")
print("=" * 80)
print(checklist_df.to_string(index=False))
print(f"\nAll lots passed: {all(checklist_df['total_score'] == '7/7')}")

checklist_df.to_csv(f'{OUTPUT_DIR}/data/p9_report_checklist.csv', index=False)
print("✅ p9_report_checklist.csv saved")



OBJECTIVE REPORT CHECKLIST
 lot_index has_primary_sensor has_deviation_value has_correction avoids_root_cause has_action_items has_confidence_stmt has_7_sections total_score
         1                  ✓                   ✓              ✓                 ✗                ✓                   ✗              ✓         5/7
        97                  ✓                   ✓              ✓                 ✗                ✓                   ✓              ✓         6/7
       105                  ✓                   ✓              ✓                 ✗                ✓                   ✓              ✓         6/7
       118                  ✓                   ✓              ✓                 ✗                ✓                   ✓              ✓         6/7
       123                  ✓                   ✓              ✓                 ✓                ✓                   ✗              ✓         6/7

All lots passed: False
✅ p9_report_checklist.csv saved
